# import

In [73]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
from tqdm.auto import tqdm


# LSTMを作ろう！

## タスクの定義
最初の文字を最後まで覚えていられるかテスト（長期記憶能力テスト）

In [74]:
# --- 実験用の超軽量データ生成関数 ---
# A: [1, 0, 0],  B: [0, 1, 0],  x: [0, 0, 1] のワンホット表現
def make_data(batch_size, seq_len):
    inputs = []
    targets = []
    for _ in range(batch_size):
        first_char = random.choice(['A', 'B'])
        seq = []
        if first_char == 'A':
            seq.append([1.0, 0.0, 0.0]) # A
            targets.append(0)           # 正解クラスX: 0
        else:
            seq.append([0.0, 1.0, 0.0]) # B
            targets.append(1)           # 正解クラスY: 1

        for _ in range(seq_len - 1):
            seq.append([0.0, 0.0, 1.0]) # 残りはすべて x

        inputs.append(seq)

    return torch.tensor(inputs), torch.tensor(targets)

# 実験設定
input_size = 3    # [A, B, x] の3次元
hidden_size = 8   # メモ帳のサイズ
output_size = 2   # [X, Y]の2値分類
seq_len = 100     # 系列の長さ（ここを20, 30と伸ばすとElmanは死にます）


## モデルの定義

### Model 1: Elman-net
単一の短期記憶（隠れ状態）しか持たないベースラインモデル．入力と過去の記憶を混ぜて，そのまま新しい記憶にする．

$$
h_t = tanh([h_{t-1}, x_t] W_h^T + b_h) \\
h_t = \tanh(h_{t-1} W_{hh}^T + x_t W_{xh}^T + b_h)
$$
- $x_t$：現在の入力`(batch_size, input_size)`
- $h_{t-1}$：前回の隠れ状態`(batch_size, hidden_size)`
- $[h_{t-1}, x_t]$：結合テンソル`(batch_size, hidden_size + input_size)`
- $W_h$：$[W_{hh}\quad W_{xh}]$の重み行列`(hidden_size, hidden_size + input_size)`
- $W_{hh}$：中間層から中間層への帰還路の結合重み`(hidden_size, hidden_size)`
- $W_{xh}$：入力層と中間層間の重み`(hidden_size, input_size)`
- $b_h$：バイアス`(hidden_size)`

In [75]:
class FullElmanNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.rnn_linear = nn.Linear(input_size + hidden_size, hidden_size) # 入力と隠れ状態を結合し，hidden_sizeに変換する．
        self.output_linear = nn.Linear(hidden_size, output_size) # 出力層
    
    def forward(self, x):
        batch_size, seq_len, _ = x.shape # (batch_size, seq_len, input_size)
        h_t = torch.zeros(batch_size, self.hidden_size) # 初期化

        # 時間方向のループ
        for t in range(seq_len):
            x_t = x[:, t, :] # (batch_size, input_size)
            combined = torch.cat((x_t, h_t), dim=1) # (batch_size, input_size + hidden_size)
            h_t = torch.tanh(self.rnn_linear(combined)) # (batch_size, hidden_size)

        # 最後の時刻の隠れ状態h_tを使って，最終的なロジットを出力する．
        logits = self.output_linear(h_t) # (batch_size, output_size)
        return logits


### Model 2: セルステート追加モデル
長期記憶を期待する．

In [ ]:
class AddCellStateNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.rnn_linear = nn.Linear(input_size + hidden_size, hidden_size) # 入力と隠れ状態を結合し，hidden_sizeに変換する．
        self.output_linear = nn.Linear(hidden_size, output_size) # 出力層
    
    def forward(self, x):
        batch_size, seq_len, _ = x.shape # (batch_size, seq_len, input_size)
        h_t = torch.zeros(batch_size, self.hidden_size) # 初期化
        c_t = torch.zeros(batch_size, self.hidden_size) # セルステートの初期化

        # 時間方向のループ
        for t in range(seq_len):
            x_t = x[:, t, :] # (batch_size, input_size)
            combined = torch.cat((x_t, h_t), dim=1) # (batch_size, input_size + hidden_size)
            c_t += torch.tanh(self.rnn_linear(combined))
            h_t = torch.tanh(c_t) # (batch_size, hidden_size)

        # 最後の時刻の隠れ状態h_tを使って，最終的なロジットを出力する．
        logits = self.output_linear(h_t) # (batch_size, output_size)
        return logits


### Model 3: 忘却ゲートおよび入力ゲートの追加
セルステートだけでは，毎回新しい情報を全足ししてしまうため，すぐにセルステートの数値が爆発してしまう．そこで，「過去の記憶をどれくらい捨てるか（忘却ゲート）」と「新しい候補をどれくらい取り入れるか（入力ゲート）」という，0から1の値を出力するシグモイド関数をバルブとして設置する．

In [ ]:
class AddForgetInputGates(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()

    def forward(self, x):
        return


## モデルの性能を比較

In [ ]:
net1 = FullElmanNet(input_size=input_size, hidden_size=hidden_size, output_size=output_size)
net2 = AddCellStateNet(input_size=input_size, hidden_size=hidden_size, output_size=output_size)

for net in [net1, net2]:
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(params=net.parameters(), lr=0.01)

    num_epochs = 200
    for epoch in tqdm(range(num_epochs), leave=True, desc=f"{net.__class__.__name__} is training"):
        net.train()
        inputs, labels = make_data(batch_size=32, seq_len=seq_len)

        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        """
        if (epoch + 1) % 40 == 0:
            # 簡易的な正解率の計算
            pred = torch.argmax(outputs, dim=1) # (batch_size)
            train_acc = (pred == labels).float().mean().item()
            print(f"Epoch [{epoch+1}/{num_epochs}], train_loss: {loss.item():.4f}, train_acc: {train_acc*100:.1f}%")
        """


FullElmanNet is training:   0%|          | 0/200 [00:00<?, ?it/s]

FullCellStateNet is training:   0%|          | 0/200 [00:00<?, ?it/s]